# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melikekaya01/flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### My answer

My lane is primarily a **ranking / scoring task**.

The goal is to assign each eligible content page a CTR-opportunity score and rank the pages from highest to lowest review priority.

This is a ranking task because the main decision is not simply whether a page has a low CTR. The practical decision is which pages an SEO or content editor should inspect first when review time is limited.

The final output will be a ranked review queue. Each row will represent one pseudonymized content page and will include an opportunity score and reason codes explaining why the page was ranked highly.

The output supports actions such as improving the title or meta description, checking search-intent alignment, updating content, improving snippet structure, monitoring the page, or leaving it unchanged.


In [6]:
task_type = "Ranking / Scoring"
decision = "Which visible content pages should an SEO reviewer inspect first?"
output = "A ranked CTR-opportunity queue with scores and reason codes"

print("Task type:", task_type)
print("Decision:", decision)
print("Output:", output)

Task type: Ranking / Scoring
Decision: Which visible content pages should an SEO reviewer inspect first?
Output: A ranked CTR-opportunity queue with scores and reason codes


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### My answer

For this starter-data exercise, I will use a defined binary proxy called `ctr_opportunity_proxy`.

A page will receive a proxy value of `1` when all of the following conditions are met:

- it has at least 500 impressions in the observed 90-day period,
- it has valid average-position information between 1 and 20,
- and its observed CTR is below 0.5%.

All other pages will receive a proxy value of `0`.

This proxy represents a visible page with weak observed CTR that may deserve human review.

It is important to call this a proxy rather than a perfect ground-truth target. The label is created from a transparent rule in the current dataset. It does not prove that the page is genuinely poor, that its title is the cause, or that editing the page will improve performance.

A stronger future target would come from an observed later outcome, such as whether a reviewed page gains clicks or CTR in a future measurement window after controlling for search position and traffic volume.

In [7]:
import pandas as pd

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "melikekaya01/flyrank-ml/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

df["ctr_opportunity_proxy"] = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
).astype(int)

df[
    [
        "content_id",
        "impressions_90d",
        "avg_position",
        "ctr",
        "ctr_opportunity_proxy",
    ]
].head(10)


,content_id,impressions_90d,avg_position,ctr,ctr_opportunity_proxy
0,content_304f48230142,3803,10.6,0.76,0
1,content_a1fb4e703a9e,15320,20.3,0.05,0
2,content_9aa793d4d895,12581,36.5,0.09,0
3,content_331d6c4de07b,11751,6.2,0.49,1
4,content_d99b7a2d90ca,19140,44.0,0.13,0
5,content_d4084a4bc775,3970,8.5,0.03,1
6,content_9a34b442b552,20,7.0,0.00,0
7,content_a63219c6e95a,1724,21.2,0.06,0
8,content_5e6c160719bc,32574,46.0,0.09,0
9,content_c27558df2b0c,1240,4.9,0.16,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### My answer

My primary success metric will be **Precision@50**.

Precision@50 measures the proportion of the first 50 recommended pages that have a positive `ctr_opportunity_proxy`.

I chose this metric because the final output is intended for a reviewer with limited capacity. The reviewer is more likely to inspect the first 50 pages in the queue than all pages in the dataset.

For example, a Precision@50 value of `0.80` would mean that 40 of the first 50 recommended pages meet the defined CTR-opportunity proxy.

A useful scoring method should clearly outperform the overall positive-proxy rate and a simple transparent fixed-rule baseline.

The metric evaluates the concentration of relevant pages near the top of the queue. It does not prove that editing those pages will improve traffic or CTR.

In [8]:
def precision_at_k(data, score_column, target_column, k=50):
    """
    Sort rows by the score column and calculate
    the positive-target rate among the first k rows.
    """
    top_k = data.sort_values(
        score_column,
        ascending=False
    ).head(k)

    return top_k[target_column].mean()


overall_proxy_rate = df["ctr_opportunity_proxy"].mean()

print("Evaluation metric: Precision@50")
print("Overall positive-proxy rate:", round(overall_proxy_rate, 4))
print(
    "A useful ranking should outperform this reference rate "
    "and a transparent fixed-rule baseline."
)

Evaluation metric: Precision@50
Overall positive-proxy rate: 0.3253
A useful ranking should outperform this reference rate and a transparent fixed-rule baseline.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### My answer

The unit of analysis is **one pseudonymized content page**.

Each row represents one content item belonging to one pseudonymized client.

The row contains page-level information such as impressions, clicks, CTR, average position, content type, content age, freshness, sessions, and engagement.

The page-level unit fits the decision because the final action is also page-level. A reviewer decides whether that specific page should have its title or meta description improved, whether its search-intent alignment should be checked, whether the content should be updated, or whether the page should remain unchanged.

The `content_id` and `client_id` fields are identifiers. They can support grouping, joining, and evaluation splits, but they should not be treated as predictive features.

In [11]:
eligible_pages = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
].copy()

unit_columns = [
    "content_id",
    "client_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "ctr_opportunity_proxy",
]

unit_df = eligible_pages[unit_columns].copy()

print("Unit of analysis: one pseudonymized content page")
print("Eligible rows:", len(unit_df))
print("Unique content pages:", unit_df["content_id"].nunique())
print("Duplicate content IDs:", unit_df["content_id"].duplicated().sum())

unit_df.head(10)

Unit of analysis: one pseudonymized content page
Eligible rows: 12023
Unique content pages: 12023
Duplicate content IDs: 0


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,ctr_opportunity_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,29,0.76,10.6,17,5.88,0
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,58,0.49,6.2,78,1.28,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,1,0.03,8.5,5,0.00,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,1240,2,0.16,4.9,3,0.00,1
10,content_d8ee6cc6d642,client_19581e27de,keyword article,329,104,20919,324,1.55,2.2,326,6.75,0
12,content_42fb2cad9ecf,client_6208ef0f77,keyword article,124,104,7228,127,1.76,5.6,233,3.43,0
16,content_78bd1d4a1d4d,client_6208ef0f77,keyword article,307,104,13848,21,0.15,8.9,475,0.21,1
17,content_761a44afda12,client_19581e27de,keyword article,421,22,9449,7,0.07,7.3,118,11.86,1
18,content_0b360eb9db55,client_349c41201b,keyword article,119,20,5141,7,0.14,11.4,43,2.33,1
20,content_0d748c484ab1,client_f369cb89fc,keyword article,95,20,2538,14,0.55,5.6,13,7.69,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### My answer

A fixed rule is useful as a transparent baseline, but a single threshold is unlikely to capture the full CTR-opportunity pattern.

Low CTR does not have the same meaning for every page. A page ranking near position 2 with weak CTR may represent a different opportunity from a page ranking near position 18. Impression volume also matters because a low CTR based on very few impressions may be unstable or operationally unimportant.

Content type, search intent, average position, impression volume, content age, freshness, engagement, and click volume may interact in different ways.

For example, the rule:

`impressions_90d >= 500 and avg_position <= 20 and ctr < 0.5`

creates a transparent candidate set. However, it gives every matching page the same result and does not decide which page should be reviewed first.

A scoring or machine-learning method can combine several measured signals and rank the pages according to estimated review opportunity.

ML only earns its place if it produces a better top-of-queue result than the transparent baseline on held-out data.

The output remains decision-support. It does not prove why CTR is low, predict Google's algorithm, or guarantee that changing a page will improve clicks.

In [ ]:
eligible_pages["fixed_rule_score"] = (
    (eligible_pages["ctr"] < 0.5).astype(int)
    + (eligible_pages["ctr"] < 0.25).astype(int)
    + (eligible_pages["avg_position"] <= 10).astype(int)
    + (eligible_pages["impressions_90d"] >= 1000).astype(int)
    + (eligible_pages["days_since_last_update"] >= 180).astype(int)
)

baseline_precision_50 = precision_at_k(
    data=eligible_pages,
    score_column="fixed_rule_score",
    target_column="ctr_opportunity_proxy",
    k=50,
)

print(
    "Fixed-rule Precision@50:",
    round(baseline_precision_50, 4)
)

eligible_pages[
    [
        "content_id",
        "impressions_90d",
        "avg_position",
        "ctr",
        "days_since_last_update",
        "fixed_rule_score",
        "ctr_opportunity_proxy",
    ]
].sort_values(
    "fixed_rule_score",
    ascending=False
).head(10)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.